In [0]:
from pyspark.sql.functions import *

In [0]:
df_compras = spark.table('finalproject.silver.compras')

df_detalles = spark.table('finalproject.silver.detalles')

In [0]:
df_fact_compras = df_compras\
    .join(df_detalles, 
          df_compras.factura == df_detalles.factura, 
          'inner'
    ).select(df_compras.venta_id,df_compras.factura,df_compras.fecha_orden,df_compras.fecha_envio,df_compras.estado,df_compras.metodo_pago,df_compras.grupo_dias_abierto,df_compras.cliente_id,df_compras.tipo_documento,df_compras.num_documento,df_compras.nombre_cliente,df_compras.tipo_cliente,df_compras.vendedor,df_compras.departamento,df_detalles.detalle_id,df_detalles.tienda,df_detalles.categoria,df_detalles.subcategoria,df_detalles.producto,df_detalles.unidades,df_detalles.subtotal)

In [0]:
df_fact_compras = df_fact_compras.withColumn('fecha_carga',current_timestamp())

In [0]:
try:
    df_fact_compras.write.mode('overwrite').format('delta').option('overwriteSchema',True).saveAsTable('linio.gold_fact_compras')
    
except Exception as e:
    import traceback
    # Tipo de error
    error_type = type(e).__name__
    # Descripcion del error 
    error_summary = str(e)
    # Traza del error (ver en qué parte se generó el error)
    error_trace = traceback.format_exc()

    # Error completo
    error_msg_full = f"{error_type}: {error_summary}\n{error_trace}"

    if len(error_msg_full) > 500:
        error_msg = error_msg_full[:500] + "\n[...] ERROR TRUNCADO [...]"
    else:
        error_msg = error_msg_full
    
    dbutils.jobs.taskValues.set(key = "error", value = error_msg)
    raise e